In [ ]:
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA

from data_loaders.mice import MiceDatasetLoader
from aind_analysis_arch_result_access import han_pipeline

# pd.set_option('display.max_rows', None)

## merge data df

### load session level meta data

In [ ]:
df_han = han_pipeline.get_session_table(if_load_bpod=True)
df_han.columns

In [ ]:
df_han[(df_han['subject_id'] == '687582') & (df_han['session_date'] == '2023-11-28')].iloc[0]['subject_alias']
# 687582_2023-11-28_12-53-25

In [ ]:
df_han['current_stage_actual'].unique()

In [ ]:
print(df_han.columns.tolist())

### load pre-saved trial df

In [ ]:
# read /data/mice_behavioral_data_20260309-0_200-185844.pkl
with open("/data/mice_behavioral_data_20260309-0_200-185844.pkl", "rb") as f:
    df_0_200 = pickle.load(f)
    
# read /data/mice_behavioral_data_20260309-200_400-114605.pkl
with open("/data/mice_behavioral_data_20260309-200_400-114605.pkl", "rb") as f:
    df_200_400 = pickle.load(f)
    
# read /data/mice_behavioral_data_20260309-400_600-213302.pkl
with open("/data/mice_behavioral_data_20260309-400_600-213302.pkl", "rb") as f:
    df_400_600 = pickle.load(f)

# read /data/mice_behavioral_data_20260310-600_end-120215.pkl
with open("/data/mice_behavioral_data_20260310-600_end-120215.pkl", "rb") as f:
    df_600_end = pickle.load(f)

# read /data/mice_behavioral_data_20260309-misc_778149_778147_753618-215212.pkl
with open("/data/mice_behavioral_data_20260309-misc_778149_778147_753618-215212.pkl", "rb") as f:
    df_misc = pickle.load(f)
    
all_dfs = [df_0_200, df_200_400, df_400_600, df_600_end, df_misc]

In [ ]:
# check if all dfs in all_dfs have the same columns
for idx, df in enumerate(all_dfs):
    print(f"df_{idx}: {len(df)}")
    if not df.columns.to_list().sort() == all_dfs[0].columns.to_list().sort():
        print(f"Columns do not match in {df.shape[0]} vs {all_dfs[0].shape[0]}!")


# concatenate all_dfs into one df
df_all = pd.concat(all_dfs, ignore_index=True)
print(f"df_all: {len(df_all)}")

In [ ]:
df_all.columns

In [ ]:
# check how many unique subjects and sessions are in df_all
print("Total number of subjects:", df_all['subject_id'].nunique())
print("Total number of sessions:", df_all['ses_idx'].nunique())
print("Sessions per subject:")

# sort by number of sessions per subject
print(df_all.groupby('subject_id')['ses_idx'].nunique().sort_values(ascending=False))

In [ ]:
# generate a session table with subject_id, ses_idx
df_session = df_all[['subject_id', 'ses_idx']].drop_duplicates().reset_index(drop=True)

# get session_date from ses_idx
df_session['session_date'] = df_session['ses_idx'].apply(lambda x: x.split('_')[1])

# based on subject_id and session_date, get nwb_suffix and current_stage_actual from df_han
def get_nwb_suffix_and_current_stage(subject_id, session_date):
    row = df_han[(df_han['subject_id'] == subject_id) & (df_han['session_date'] == session_date)]
    if len(row) == 0:
        print(f"Warning: no row found for subject_id {subject_id} and session_date {session_date} in df_han!")
        return None, None, None
    elif len(row) > 1:
        print(f"Warning: multiple rows found for subject_id {subject_id} and session_date {session_date} in df_han!")
        return None, None, None
    else:
        return row.iloc[0]['nwb_suffix'], row.iloc[0]['current_stage_actual'], row.iloc[0]['curriculum_name']

df_session[['nwb_suffix', 'current_stage_actual', 'curriculum_name']] = df_session.apply(
    lambda x: get_nwb_suffix_and_current_stage(x['subject_id'], x['session_date']),
    axis=1,
    result_type='expand'
)


# check where nwb_suffix is None
print(f"Total number of session not found in df_han: {len(df_session[df_session['nwb_suffix'].isnull()])}")
# df_session[df_session['nwb_suffix'].isnull()]


# drop rows where nwb_suffix is NaN
n_before = len(df_session)
df_session = df_session.dropna(subset=['nwb_suffix']).reset_index(drop=True)
n_after = len(df_session)
print(f"Dropped {n_before - n_after} rows with NaN nwb_suffix. Remaining rows: {n_after}")

In [ ]:
# check how many unique subjects and sessions are in df_session
print("Total number of subjects in df_session:", df_session['subject_id'].nunique())
print("Total number of sessions in df_session:", df_session['ses_idx'].nunique())
print("Sessions per subject in df_session:")
print(df_session.groupby('subject_id')['ses_idx'].nunique().sort_values(ascending=False))

In [ ]:
df_mature_session = df_session[
    df_session['current_stage_actual'].isin(['STAGE_FINAL', 'GRADUATED'])
].copy()

In [ ]:
# check how many unique subjects and sessions are in df_mature_session
print("Total number of subjects in df_mature_session:", df_mature_session['subject_id'].nunique())
print("Total number of sessions in df_mature_session:", df_mature_session['ses_idx'].nunique())
print("Sessions per subject in df_mature_session:")
print(df_mature_session.groupby('subject_id')['ses_idx'].nunique().sort_values(ascending=False))

In [ ]:
# print number of sessions per subject, sorted
print("Number of sessions per subject (sorted):")
print(df_session.groupby('subject_id').size().sort_values(ascending=False))

In [ ]:
print("Number of mature sessions per subject (sorted):")
print(df_mature_session.groupby('subject_id').size().sort_values(ascending=False))

In [ ]:
# print number of sessions per curriculum, sorted
df_session['curriculum_name'].value_counts().sort_values(ascending=False)

In [ ]:
# print number of sessions per curriculum, sorted
df_mature_session['curriculum_name'].value_counts().sort_values(ascending=False)

In [ ]:
# First group by curriculum_name, then group by subject_id, and print number of sessions per subject, sorted
(df_session
    .groupby(['curriculum_name', 'subject_id'])
    .size()
    .rename('n_sessions')
    .reset_index()
    .sort_values(['curriculum_name', 'n_sessions'], ascending=[True, False])
    .set_index(['curriculum_name', 'subject_id'])
)

In [ ]:
# print each curriculum separately, the subjects in that curriculum, and the number of sessions for each subject, sorted by number of sessions
curriculums = df_session['curriculum_name'].unique()
for curriculum in curriculums:
    print(f"Curriculum: {curriculum}")
    df_curriculum = df_session[df_session['curriculum_name'] == curriculum]
    print(df_curriculum.groupby('subject_id').size().rename('n_sessions').sort_values(ascending=False))
    print("\n")

In [ ]:
(df_mature_session
    .groupby(['curriculum_name', 'subject_id'])
    .size()
    .rename('n_sessions')
    .reset_index()
    .sort_values(['curriculum_name', 'n_sessions'], ascending=[True, False])
    .set_index(['curriculum_name', 'subject_id'])
)

In [ ]:
# print each curriculum separately, the subjects in that curriculum, and the number of sessions for each subject, sorted by number of sessions
curriculums = df_mature_session['curriculum_name'].unique()
for curriculum in curriculums:
    print(f"Curriculum: {curriculum}")
    df_curriculum = df_mature_session[df_mature_session['curriculum_name'] == curriculum]
    print(df_curriculum.groupby('subject_id').size().rename('n_sessions').sort_values(ascending=False))
    print("\n")

In [ ]:
# check how many subjects contain more than 10 mature_sessions
n_session_threshold = 10
mature_sessions_per_subject = df_mature_session.groupby('subject_id').size()
n_subjects_enough = (mature_sessions_per_subject > n_session_threshold).sum()

print(f"Number of subjects with >10 mature sessions: {n_subjects_enough}")
print("Subject IDs with >10 mature sessions:")
print(mature_sessions_per_subject[mature_sessions_per_subject > n_session_threshold].sort_values(ascending=False))

In [25]:
# get the list of subject_ids of the top k subjects with most mature sessions
# per curriculum in ['Uncoupled Without Baiting', 'Uncoupled Baiting', 'Coupled Baiting']

curricula = ['Uncoupled Without Baiting', 'Uncoupled Baiting', 'Coupled Baiting']
k = 6


top_k_per_curriculum = (
    df_mature_session[df_mature_session['curriculum_name'].isin(curricula)]
    .groupby(['curriculum_name', 'subject_id'])
    .size()
    .rename('n_mature_sessions')
    .reset_index()
    .sort_values(['curriculum_name', 'n_mature_sessions'], ascending=[True, False])
    .groupby('curriculum_name')
    .head(k)
    .set_index(['curriculum_name', 'subject_id'])
)

print(top_k_per_curriculum)

top_k_subject_ids = (
    top_k_per_curriculum
    .reset_index()
    .groupby('curriculum_name')['subject_id']
    .apply(list)
)
print("\nTop " + str(k) + " subject_ids per curriculum:")
print(top_k_subject_ids)


                                      n_mature_sessions
curriculum_name           subject_id                   
Coupled Baiting           786868                     18
                          708027                     11
                          705599                      8
                          722989                      8
                          708031                      5
                          700708                      4
Uncoupled Baiting         792292                     65
                          801995                     65
                          802252                     64
                          801997                     61
                          792288                     55
                          791174                     52
Uncoupled Without Baiting 791094                     58
                          778234                     47
                          786867                     45
                          791096                

In [26]:
# flatten the list of subject_ids in top_k_subject_ids into a single list of unique subject_ids
train_set_ids = []
test_set_ids = []
for subject_list in top_k_subject_ids:
    train_set_ids.extend(subject_list[:k//2])
    test_set_ids.extend(subject_list[k//2:k])

print("Unique subject_ids in top " + str(k) + " per curriculum:")
print("Train set subject_ids:", train_set_ids)
print("Test set subject_ids:", test_set_ids)

Unique subject_ids in top 6 per curriculum:
Train set subject_ids: ['786868', '708027', '705599', '792292', '801995', '802252', '791094', '778234', '786867']
Test set subject_ids: ['722989', '708031', '700708', '801997', '792288', '791174', '791096', '784803', '790768']


In [ ]:
# select rows in df_all where subject_id is in train_set_ids or test_set_ids
# retain only the selected columns

cols_to_retain = ['trial', 'subject_id', 'ses_idx', 'animal_response', 'earned_reward']

df_train = df_all[df_all['subject_id'].isin(train_set_ids)][cols_to_retain].copy()
df_test = df_all[df_all['subject_id'].isin(test_set_ids)][cols_to_retain].copy()

In [ ]:
df_train

In [ ]:
def plot_session_distribution(
    df,
    subject_col='subject_id',
    figsize=(12, 4),
    dpi=150,
    xtick_step=5,
    title_suffix=''
 ):
    sessions_per_subject = df.groupby(subject_col).size()
    if sessions_per_subject.empty:
        raise ValueError('No rows available to plot session distribution.')

    n_sessions_sorted = sessions_per_subject.sort_values(ascending=False)
    x_min, x_max = sessions_per_subject.min(), sessions_per_subject.max()
    bins = np.arange(x_min, x_max + 2) - 0.5

    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=figsize, dpi=dpi)

    # left: histogram of sessions per subject
    axs[0].hist(sessions_per_subject, bins=bins, edgecolor='black', alpha=0.8)
    axs[0].set_xlabel('n_sessions')
    axs[0].set_ylabel('Number of subjects')
    axs[0].set_title(f'Distribution of n_sessions per subject{title_suffix}')
    axs[0].set_xticks(np.arange(x_min, x_max + 1, xtick_step))

    # right: cumulative total sessions while adding subjects from most to least sessions
    cum_sessions = n_sessions_sorted.cumsum()
    subject_rank = np.arange(1, len(cum_sessions) + 1)
    axs[1].step(subject_rank, cum_sessions.values, where='post')
    axs[1].set_xlabel('Number of subjects included (ranked by n_sessions, desc)')
    axs[1].set_ylabel('Cumulative number of sessions')
    axs[1].set_yscale('log')
    axs[1].set_title(f'Cumulative sessions by subject rank{title_suffix}')
    axs[1].set_xlim(1, len(cum_sessions))
    axs[1].grid(alpha=0.3)

    fig.tight_layout()
    return fig, axs

In [ ]:
fig, axs = plot_session_distribution(df_session)
plt.show()

In [ ]:
fig, axs = plot_session_distribution(df_mature_session, title_suffix='mature_only')
plt.show()

## test data loader for mice model

In [ ]:
# uncoupled no baiting
# subject_ids = [774212]
# subject_ids = [793446]
# subject_ids = [711041]
# subject_ids = [722832]
# subject_ids = [746346]
# subject_ids = [758017]  # takes 23m to load 
# subject_ids = [751766]
# subject_ids = [714116]  # error when loading
# subject_ids = [727456]
# subject_ids = [751765]
# 751769
# 752014
# 754897


# uncoupled baiting
# subject_ids = [790724]
# subject_ids = [730942]
# subject_ids = [739195]  # load 28m 
# subject_ids = [729674]
# subject_ids = [743794]
# subject_ids = [720956]
# subject_ids = [728570]
# subject_ids = [728568]
# subject_ids = [713377], none
# subject_ids = [716870], none


# mixed
# subject_ids = [774212, 790724]
# subject_ids = [793446, 790724]
subject_ids = [751766, 727456, 751765, 729674, 728570, 739195]


loader = MiceDatasetLoader(
    subject_ids=subject_ids,
    ignore_policy='include',
    features={'animal_response': 'prev choice', 'rewarded': 'prev reward'},
    eval_every_n=2,
    multisubject=False,
    seed=42,
)
dataset_bundle = loader.load()

In [ ]:
dataset_bundle

In [ ]:
# 1. Pickle the object to a file
with open('./dataset_bundle_test-mature_only-ignore_include.pkl', 'wb') as fo:
    pickle.dump(dataset_bundle, fo)

In [ ]:
# 2. Unpickle the object from the file
with open('./dataset_bundle_test-mature_only-ignore_include.pkl', 'rb') as fi:
    loaded_obj = pickle.load(fi)

## plot latents from trained dis-rnn

In [ ]:
from disentangled_rnns.library import disrnn, plotting, rnn_utils

In [ ]:
fig_dpi = 300

def plot_traj_in_PC_space(
    transformed_RNN_hidden_states_background,
    background_coloring_array,
    background_colorbar_label,
    background_coloring_minmax,
    num_pcs2plot,
    transformed_RNN_hidden_states_example_run=None,
    transformed_fixed_points=None,
    fixed_points_coloring_array=None,
    fixed_points_condition=None
):
    """
    Plot the transformed hidden states in the PCA space.
    Args:
        transformed_RNN_hidden_states_background: Transformed RNN hidden states for background
        transformed_RNN_hidden_states_example_run: Transformed RNN hidden states for an example run
        coloring_array: Array to color each state
        num_pcs2plot: Number of principal components to plot
        
    Returns:
        fig: matplotlib figure
    """
    n_axs = num_pcs2plot * (num_pcs2plot-1) / 2
    n_cols = num_pcs2plot
    n_rows = int(n_axs/ n_cols) + (n_axs% n_cols > 0)
    
    if n_rows == 1:
        n_rows = 2

    fig, axs = plt.subplots(
        nrows=n_rows, ncols=n_cols,
        figsize=(6*n_cols, 4*n_rows), dpi=fig_dpi
    )

    ax_id = 0
    for pc_x in range(num_pcs2plot):
        for pc_y in range(num_pcs2plot):
            if pc_x < pc_y:
                ax = axs[int(ax_id/n_cols), ax_id%n_cols]
                
                # set the title
                if transformed_fixed_points is not None:
                    ax.set_title(f'Fixed points under {fixed_points_condition}', fontsize=18)
                else:
                    ax.set_title(f'Neural space trajectoy', fontsize=18)
                
                # Plot the background
                scatter = ax.scatter(
                    transformed_RNN_hidden_states_background[:, :, pc_x],
                    transformed_RNN_hidden_states_background[:, :, pc_y],
                    s=2.5,
                    c=background_coloring_array, cmap=cm.coolwarm,
                    vmin=background_coloring_minmax[0], 
                    vmax=background_coloring_minmax[1],
                    alpha=0.9
                )
                ax.set_xlabel(f'PC {pc_x}', fontsize=18)
                ax.set_ylabel(f'PC {pc_y}', fontsize=18)
                ax.tick_params(axis='both', which='major', labelsize=12)
                fig.colorbar(
                    scatter, ax=ax,
                    label=background_colorbar_label
                )
                
                # Plot the trajectory of the example run
                if transformed_RNN_hidden_states_example_run is not None:
                    ax.plot(
                        transformed_RNN_hidden_states_example_run[:, pc_x],
                        transformed_RNN_hidden_states_example_run[:, pc_y],
                        color='k', lw=1, alpha=0.9
                    )
                    exameple_run_len = transformed_RNN_hidden_states_example_run.shape[0]
                    example_run = ax.scatter(
                        transformed_RNN_hidden_states_example_run[:, pc_x],
                        transformed_RNN_hidden_states_example_run[:, pc_y],
                        s=4,
                        c=np.arange(exameple_run_len), cmap=cm.copper,
                        vmin=0, vmax=exameple_run_len
                    )  # color coded by time step
                    fig.colorbar(
                        example_run, ax=ax,
                        label='Time step'
                    )

                # Plot the fixed points
                if transformed_fixed_points is not None:
                    scatter_fp = ax.scatter(
                        transformed_fixed_points[:, pc_x],
                        transformed_fixed_points[:, pc_y],
                        marker='o', s=5,
                        c=fixed_points_coloring_array, cmap=cm.gray,
                        vmin=1e-13, vmax=10, norm='log',
                        alpha=0.6
                    )  # color coded by q values
                    fig.colorbar(
                        scatter_fp, ax=ax,
                        label='Kinetic Energy (q value)'
                    )

                ax_id += 1

    fig.tight_layout()
    return fig


def plot_traj_in_PC_space_by_outcome(
    transformed_RNN_hidden_states,
    prev_choices,
    prev_rewards,
    num_pcs2plot,
    transformed_RNN_hidden_states_example_run=None,
    example_run_choices=None,
    example_run_rewards=None
):
    """
    Plot the transformed hidden states in PCA space, colored by previous outcome.
    
    Args:
        transformed_RNN_hidden_states: Transformed RNN hidden states [n_trials, n_sessions, n_pcs]
        prev_choices: Previous choices [n_trials, n_sessions], 0=left, 1=right
        prev_rewards: Previous rewards [n_trials, n_sessions], 0=unrewarded, 1=rewarded
        num_pcs2plot: Number of principal components to plot
        transformed_RNN_hidden_states_example_run: Optional example trajectory
        example_run_choices: Choices for example run
        example_run_rewards: Rewards for example run
        
    Returns:
        fig: matplotlib figure
    """
    # Create outcome categories: 
    # 0=rewarded_left, 1=unrewarded_left, 2=rewarded_right, 3=unrewarded_right
    outcome_categories = np.zeros_like(prev_choices)
    outcome_categories[(prev_choices == 0) & (prev_rewards == 1)] = 0  # rewarded left
    outcome_categories[(prev_choices == 0) & (prev_rewards == 0)] = 1  # unrewarded left
    outcome_categories[(prev_choices == 1) & (prev_rewards == 1)] = 2  # rewarded right
    outcome_categories[(prev_choices == 1) & (prev_rewards == 0)] = 3  # unrewarded right
    
    # Print verification statistics
    print("Outcome category counts:")
    print(f"  Rewarded left (0): {np.sum(outcome_categories == 0)}")
    print(f"  Unrewarded left (1): {np.sum(outcome_categories == 1)}")
    print(f"  Rewarded right (2): {np.sum(outcome_categories == 2)}")
    print(f"  Unrewarded right (3): {np.sum(outcome_categories == 3)}")
    
    # Define distinct colors for each outcome
    colors = {
        0: 'green',      # rewarded left
        1: 'lightcoral', # unrewarded left
        2: 'blue',       # rewarded right
        3: 'orange'      # unrewarded right
    }
    labels = {
        0: 'Rewarded Left',
        1: 'Unrewarded Left',
        2: 'Rewarded Right',
        3: 'Unrewarded Right'
    }
    
    n_axs = num_pcs2plot * (num_pcs2plot-1) / 2
    n_cols = num_pcs2plot
    n_rows = int(n_axs / n_cols) + (n_axs % n_cols > 0)
    
    if n_rows == 1:
        n_rows = 2

    fig, axs = plt.subplots(
        nrows=n_rows, ncols=n_cols,
        figsize=(5*n_cols, 5*n_rows), dpi=fig_dpi
    )

    ax_id = 0
    for pc_x in range(num_pcs2plot):
        for pc_y in range(num_pcs2plot):
            if pc_x < pc_y:
                ax = axs[int(ax_id/n_cols), ax_id%n_cols]
                
                ax.set_title(f'Colored by previous outcome', fontsize=18)
                
                # Plot each outcome category separately
                for cat in range(4):
                    mask = (outcome_categories == cat)
                    if np.any(mask):
                        ax.scatter(
                            transformed_RNN_hidden_states[:, :, pc_x][mask],
                            transformed_RNN_hidden_states[:, :, pc_y][mask],
                            s=2.5,
                            c=colors[cat],
                            label=labels[cat],
                            alpha=0.6
                        )
                
                ax.set_xlabel(f'PC {pc_x}', fontsize=18)
                ax.set_ylabel(f'PC {pc_y}', fontsize=18)
                ax.tick_params(axis='both', which='major', labelsize=12)
                
                # Add legend only to first subplot
                if ax_id == 0:
                    ax.legend(fontsize=12, markerscale=4, loc='best')
                
                # Plot example trajectory if provided
                if transformed_RNN_hidden_states_example_run is not None:
                    # Plot trajectory line
                    ax.plot(
                        transformed_RNN_hidden_states_example_run[:, pc_x],
                        transformed_RNN_hidden_states_example_run[:, pc_y],
                        color='k', lw=2, alpha=0.5, zorder=100
                    )
                    
                    # Plot example run points colored by outcome
                    if example_run_choices is not None and example_run_rewards is not None:
                        example_categories = np.zeros_like(example_run_choices)
                        example_categories[(example_run_choices == 0) & (example_run_rewards == 1)] = 0
                        example_categories[(example_run_choices == 0) & (example_run_rewards == 0)] = 1
                        example_categories[(example_run_choices == 1) & (example_run_rewards == 1)] = 2
                        example_categories[(example_run_choices == 1) & (example_run_rewards == 0)] = 3
                        
                        for cat in range(4):
                            mask = (example_categories == cat)
                            if np.any(mask):
                                ax.scatter(
                                    transformed_RNN_hidden_states_example_run[:, pc_x][mask],
                                    transformed_RNN_hidden_states_example_run[:, pc_y][mask],
                                    s=20,
                                    c=colors[cat],
                                    edgecolors='black',
                                    linewidths=0.5,
                                    alpha=0.9,
                                    zorder=101
                                )

                ax_id += 1

    fig.tight_layout()
    return fig


# Verification - test the function
def verify_outcome_categorization(choices, rewards, n_trials_to_check=10):
    """Helper function to verify outcome categorization is correct."""
    print("\nVerification of first few trials:")
    print("Trial | Choice | Reward | Expected Category | Actual Category")
    print("-" * 70)
    
    for i in range(min(n_trials_to_check, len(choices))):
        choice = choices[i]
        reward = rewards[i]
        
        # Expected category
        if choice == 0 and reward == 1:
            expected = "Rewarded Left (0)"
            expected_val = 0
        elif choice == 0 and reward == 0:
            expected = "Unrewarded Left (1)"
            expected_val = 1
        elif choice == 1 and reward == 1:
            expected = "Rewarded Right (2)"
            expected_val = 2
        elif choice == 1 and reward == 0:
            expected = "Unrewarded Right (3)"
            expected_val = 3
        else:
            expected = "Invalid"
            expected_val = -1
        
        # Actual category
        outcome_cat = np.zeros(1, dtype=int)
        if choice == 0 and reward == 1:
            outcome_cat[0] = 0
        elif choice == 0 and reward == 0:
            outcome_cat[0] = 1
        elif choice == 1 and reward == 1:
            outcome_cat[0] = 2
        elif choice == 1 and reward == 0:
            outcome_cat[0] = 3
            
        match = "✓" if outcome_cat[0] == expected_val else "✗"
        print(f"{i:5d} | {choice:6.0f} | {reward:6.0f} | {expected:25s} | {outcome_cat[0]} {match}")

In [ ]:
import json

# uncoupled no baiting
# trained_model_fp = '/root/capsule/data/results-mice-uncoupled_no_baiting-793446-95876cab-5c77-447c-95c9-e82fcc42b87e'
# model_id = '11'

# uncoupled baiting
# trained_model_fp = '/root/capsule/data/results-mice-uncoupled_baiting-790724-051437f2-9c43-4576-94ea-e9a524671b86'
# model_id = '9', '10'

# mixed
## 5k
# trained_model_fp = '/root/capsule/data/results-mice-mixed-793446_790724-8942cd80-aef7-426b-aefb-31f48fc982ff'
# model_id = '10'
## 20k
trained_model_fp = '/root/capsule/data/results-mice-mixed-793446_790724-20k-a9407158-af57-4160-831a-b715915ddd10'
# model_id = '10'


model_id = '10'


with open(
    f'{trained_model_fp}/{model_id}/inputs/jobs/.hydra/config.json', 
    'r'
) as fp:
    disrnn_config_json = json.load(fp)

with open(
    f'{trained_model_fp}/{model_id}/outputs/params.json', 
    'r'
) as fp:
    params = rnn_utils.to_np(json.load(fp))


# params_disrnn = params
params_disrnn = params['hk_disentangled_rnn']

In [ ]:
for key in params.keys():
    print(key)

In [ ]:
dataset_eval = dataset_bundle.eval_set

disrnn_config = disrnn.DisRnnConfig(
    # Dataset related
    obs_size=2,  # Choice, reward
    output_size=2,  # Choose left / choose right
    x_names=dataset_eval.x_names,
    y_names=dataset_eval.y_names,
    # Network architecture
    latent_size=disrnn_config_json['model']['architecture']['latent_size'],
    update_net_n_units_per_layer=disrnn_config_json['model']['architecture']['update_net_n_units_per_layer'],
    update_net_n_layers=disrnn_config_json['model']['architecture']['update_net_n_layers'],
    choice_net_n_units_per_layer=disrnn_config_json['model']['architecture']['choice_net_n_units_per_layer'],
    choice_net_n_layers=disrnn_config_json['model']['architecture']['choice_net_n_layers'],
    activation=disrnn_config_json['model']['architecture']['activation'],
    # Penalties
    # noiseless_mode=False,
    noiseless_mode=True,
    latent_penalty=disrnn_config_json['model']['penalties']['latent_penalty'],
    choice_net_latent_penalty=disrnn_config_json['model']['penalties']['choice_net_latent_penalty'],
    update_net_obs_penalty=disrnn_config_json['model']['penalties']['update_net_obs_penalty'],
    update_net_latent_penalty=disrnn_config_json['model']['penalties']['update_net_latent_penalty'],
    l2_scale=0,
)

In [ ]:
xs, ys = dataset_eval.get_all()
print(f'xs shape: {xs.shape}, ys shape: {ys.shape}')

network_output, network_states = rnn_utils.eval_network(
    lambda: disrnn.HkDisentangledRNN(disrnn_config), params, xs)

# Compute normalized likelihood
logits = network_output[:,:,:2]  # First n_actions elements of network output are the logits (the final one is the penalty)
normalized_likelihood = rnn_utils.normalized_likelihood(labels = ys, output_logits=logits)
print(f'Normalized likelihood: {100*normalized_likelihood:.2f}%')

In [ ]:
# Plot network activations on an example session
example_session = 1

choices = xs[:, example_session, 0]
rewards = xs[:, example_session, 1]
scalars = network_states[:, example_session, :]

print(f'Choices: {choices}')
print(f'Rewards: {rewards}')

# two_armed_bandits.plot_2ab_sessdata(choices,
#                                     rewards,
#                                     scalars=scalars,
#                                     scalar_types='agent_states',
#                                     show_legend=False)
# plt.plot(scalars)

In [ ]:
# plot choice/reward history and latents
open_latents = [0,2,4]


# Filter out padding (where choices or rewards are -1)
valid_mask = (choices != -1) & (rewards != -1)
valid_indices = np.where(valid_mask)[0]
choices_filtered = choices[valid_mask]
rewards_filtered = rewards[valid_mask]
scalars_filtered = scalars[valid_mask]

fig, axs = plt.subplots(
    nrows=3, ncols=1, 
    figsize=(12, 10), 
    dpi=fig_dpi, sharex=True
)

# Top subplot: choice/reward history
ax = axs[0]
choice_mask = (choices_filtered == 0) | (choices_filtered == 1)
non_choice_mask = ~choice_mask
rewarded_mask = (rewards_filtered == 1) & choice_mask
unrewarded_mask = (rewards_filtered == 0) & choice_mask

ax.scatter(
    valid_indices[non_choice_mask],
    choices_filtered[non_choice_mask],
    marker='|',
    s=800,
    color='gray',
    label='No choice',
    alpha=0.7
)
ax.scatter(
    valid_indices[unrewarded_mask],
    choices_filtered[unrewarded_mask],
    marker='|',
    s=400,
    color='black',
    label='Unrewarded',
    alpha=0.7
)
ax.scatter(
    valid_indices[rewarded_mask],
    choices_filtered[rewarded_mask],
    marker='|',
    s=800,
    color='black',
    label='Rewarded',
    alpha=0.7
)
ax.set_ylim(-0.2, 1.2)
ax.set_yticks([0,1])
ax.set_yticklabels(['Left', 'Right'])
ax.set_title('Choices history', fontsize=18)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend(fontsize=12)

# middle subplot: latent values
ax = axs[1]
for latent_idx in open_latents:
    ax.plot(valid_indices, scalars_filtered[:, latent_idx], label=f'latent {latent_idx}')
ax.set_title('Open latents', fontsize=18)
ax.set_xlabel('Trial', fontsize=18)
ax.set_ylabel('Latent value', fontsize=18)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend(fontsize=12)

# bottom subplot: all latents
ax = axs[2]
all_latents = [0,1,2,3,4]
for latent_idx in all_latents:
    ax.plot(valid_indices, scalars_filtered[:, latent_idx], label=f'latent {latent_idx}')
ax.set_title('All latents', fontsize=18)
ax.set_xlabel('Trial', fontsize=18)
ax.set_ylabel('Latent value', fontsize=18)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend(fontsize=12)

fig.tight_layout()

In [ ]:
logit_sum = np.sum(np.exp(logits), axis=2)
action_prob = np.exp(logits)/ np.stack((logit_sum, logit_sum), axis=2)

In [ ]:
n_sessions = network_states.shape[1]

fig = plot_traj_in_PC_space(
    network_states[:,:,open_latents],
    action_prob[:,:,1],
    'a1 prob',
    background_coloring_minmax=[0,1],
    num_pcs2plot=len(open_latents),
    transformed_RNN_hidden_states_example_run=network_states[:30,example_session,open_latents],
    transformed_fixed_points=None,
    fixed_points_coloring_array=None,
    fixed_points_condition=None
)

In [ ]:
# Extract previous choices and rewards from the input data
prev_choices = xs[:, :, 0]  # Shape: [n_trials, n_sessions]
prev_rewards = xs[:, :, 1]  # Shape: [n_trials, n_sessions]

# Optional: verify the outcome categorization on one session
verify_outcome_categorization(
    choices=prev_choices[:, 0],
    rewards=prev_rewards[:, 0],
    n_trials_to_check=10
)

# Plot trajectories colored by outcome
fig = plot_traj_in_PC_space_by_outcome(
    transformed_RNN_hidden_states=network_states[:,:,open_latents],
    prev_choices=prev_choices,
    prev_rewards=prev_rewards,
    num_pcs2plot=len(open_latents),
    transformed_RNN_hidden_states_example_run=network_states[:30, example_session, open_latents],
    example_run_choices=prev_choices[:30, example_session],
    example_run_rewards=prev_rewards[:30, example_session]
)

In [ ]:
# run pca on scalars and plot the first two pcs

# PCA using training set
pca_model = PCA(n_components=5)
pca_model.fit(network_states.reshape(-1, 5))
print(f'fitted disrnn, PCA explained variance (training): {pca_model.explained_variance_ratio_}')

# plot
fig, axs = plt.subplots(
    nrows=1, ncols=2,
    figsize=(6, 3), dpi=300
)

ax = axs[0]
ax.plot(pca_model.explained_variance_ratio_)
ax.set_title('fitted disrnn')
ax.set_xlabel('PC')
ax.set_ylabel('var. exp.')

ax = axs[1]
ax.plot(np.cumsum(pca_model.explained_variance_ratio_))
ax.set_title('cumulative var. explained')
ax.set_xlabel('PC')
ax.set_ylabel('cumulative var. exp.')
ax.set_ylim(0, 1)

fig.tight_layout()


In [ ]:
# PCA transformed training set
n_sessions = network_states.shape[1]
transformed_network_states = pca_model.transform(
    network_states.reshape(-1, 5)).\
    reshape(-1, n_sessions, 5)


# plt.scatter(
#     transformed_network_states[:,:,0],
#     transformed_network_states[:,:,1],
#     marker='o', s=5, alpha=0.85
# )
# plt.plot(
#     transformed_network_states[:160,0,0],
#     transformed_network_states[:160,0,1],
#     color='k', lw=1
# )


fig = plot_traj_in_PC_space(
    transformed_network_states,
    action_prob[:,:,1],
    'a1 prob',
    background_coloring_minmax=[0,1],
    num_pcs2plot=5,
    transformed_RNN_hidden_states_example_run=transformed_network_states[:160,5,:],
    transformed_fixed_points=None,
    fixed_points_coloring_array=None,
    fixed_points_condition=None
)